# Basic mass-univariate GLMs: application v inference

This notebook analyses the first full dataset collected from a single individual, `sub-03`. The `3p0mm64ch` sequence was used throughout the experiment and functional data are available either as raw or NORDIC-denoised.

The GLM for each reconstruction computes four contrasts:
- `inference_gt_application`: `inference - application`
- `task_gt_baseline`: `(inference + application) / 2` &mdash; either trial type vs. implicit baseline
- `inference_gt_baseline`: `inference` vs. implicit baseline
- `application_gt_baseline`: `application` vs. implicit baseline

Maps are thresholded at **p < .001, uncorrected,
one-tailed** (`height_control="fpr"`, `two_sided=False` in
`nilearn.glm.threshold_stats_img`) &mdash; no multiple-comparisons correction,
as requested.

Runs to include are discovered dynamically from which `_events.tsv` files
actually exist (a run with no matched behavioural log, e.g. an aborted scan,
has no events file and is silently skipped). This stands in for the scan
manifest CSV described in `CLAUDE.md`, which doesn't exist yet.

Preprocessed BOLD is fMRIPrep's `space-MNI152NLin2009cAsym_res-2` output.
Confounds are the standard 24 motion parameters, raw CSF/WM signal, and
fMRIPrep's cosine high-pass regressors, selected explicitly by column name
from each run's `desc-confounds_timeseries.tsv` (resolved via `pybids`, not
`nilearn.interfaces.fmriprep.load_confounds`'s own file lookup: that lookup
globs for a confounds file by stripping `space`/`desc`/`res` etc. from the
BOLD filename, and since it also strips `rec-nordic`, it matches *both* the
raw and NORDIC confounds files for the same run and raises. No scrubbing /
volume censoring is applied). Because the cosine regressors already
high-pass filter the data, the model's own drift model is turned off
(`drift_model=None`) to avoid duplicating that filtering.

An 8 mm FWHM Gaussian smoothing kernel is applied before model fitting &mdash;
fairly strong, deliberately so given how little data each protocol has here.

Maps are viewed on the standard MNI152 anatomical template (`view_img`'s
default `bg_img`), matching the `MNI152NLin2009cAsym` analysis space.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import scipy.stats as st
from bids import BIDSLayout
from dotenv import load_dotenv
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits
from nilearn.glm.first_level import FirstLevelModel
from nilearn.masking import intersect_masks
from nilearn.plotting import view_img
from nilearn.image import index_img

load_dotenv()
bids_root = Path(os.environ["BIDS_ROOT_DIR"])

# layout = BIDSLayout(bids_root, validate=False, derivatives=True)

In [ ]:
SUBJECT = "04"
SESSION = "01"
ANAT_SESSION = "02"  # the anatomical (T1w) scan was acquired in an earlier session
TASK = "arc"
SPACE = "MNI152NLin2009cAsym"

ACQUISITIONS = ["2p0mm64ch"]
DUMMY_SECONDS = 10.

RECONSTRUCTIONS = [None, "nordic"]  # None = raw (no NORDIC)

CONTRASTS = {
    "task_gt_baseline": "(inference + application) / 2",
    "inference_gt_baseline": "inference",
    "application_gt_baseline": "application",
    "inference_gt_application": "inference - application",
    "application_gt_inference": "application - inference",
}
ALPHA = 0.01
SMOOTHING_FWHM_MM = 8.0

OUT_DIR = bids_root / "derivatives" / f"sub-{SUBJECT}" / "application_v_inference"
HTML_DIR = OUT_DIR / "htmls"
HTML_DIR.mkdir(parents=True, exist_ok=True)

## Helpers

`reconstruction` and `run` are filtered manually (rather than passed as
`layout.get(...)` keyword filters) so that `None` unambiguously means *the
entity is absent from the filename*, not *unfiltered*.

In [ ]:
MOTION_PARAMS = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]
MOTION_COLUMNS = (
    MOTION_PARAMS
    + [f"{p}_derivative1" for p in MOTION_PARAMS]
    + [f"{p}_power2" for p in MOTION_PARAMS]
    + [f"{p}_derivative1_power2" for p in MOTION_PARAMS]
)
COVARIATE_SIGNAL_COLUMNS = ["white_matter", "csf", "global_signal"]

MOTION_COLUMNS = ['trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']
COVARIATE_SIGNAL_COLUMNS = ['global_signal']


def get_runs(layout, acq, rec):
    """Return the run labels that have an events.tsv for this acquisition/reconstruction."""
    events = layout.get(
        subject=SUBJECT, session=SESSION, task=TASK, acquisition=acq,
        suffix="events", extension=".tsv",
    )
    events = [f for f in events if f.entities.get("reconstruction") == rec]
    return sorted({f.entities.get("run") for f in events}, key=lambda r: (r is not None, r or ""))


def get_single(layout, acq, rec, run, **filters):
    """Return the one BIDS file matching acq/rec/run plus extra filters, raising if not exactly one match."""
    kwargs = dict(subject=SUBJECT, session=SESSION, task=TASK, acquisition=acq, **filters)
    if run is not None:
        kwargs["run"] = run
    matches = [
        f for f in layout.get(**kwargs)
        if f.entities.get("reconstruction") == rec and f.entities.get("run") == run
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"expected exactly one match for acq={acq} rec={rec} run={run} "
            f"filters={filters}, got {len(matches)}"
        )
    return matches[0]


def get_t1w():
    """Return the MNI-normalized, preprocessed anatomical (T1w) scan used as the background for the statistical maps.

    Builds its own local BIDSLayout, same as run_combo below: it's only called
    once, so there's no need to keep a shared layout instance alive at notebook
    scope for its sake.
    """
    layout = BIDSLayout(bids_root, validate=False, derivatives=True)
    matches = layout.get(
        subject=SUBJECT, session=ANAT_SESSION, space=SPACE, res="2",
        desc="preproc", suffix="T1w", extension=".nii.gz",
    )
    if len(matches) != 1:
        raise RuntimeError(f"expected exactly one T1w match, got {len(matches)}: {matches}")
    return matches[0]


def read_confounds(confounds_tsv_path):
    """Load an fMRIPrep confounds TSV and select the motion, WM/CSF, and cosine drift columns used as nuisance regressors."""
    df = pd.read_csv(confounds_tsv_path, sep="\t")
    cosine_columns = [c for c in df.columns if c.startswith("cosine")]
    columns = MOTION_COLUMNS + COVARIATE_SIGNAL_COLUMNS + cosine_columns
    return df[columns].fillna(0)  # derivative columns are NaN on the first row


def fit_glm(layout, acq, rec):
    """Fit a first-level GLM for one acquisition/reconstruction across all its available runs."""
    runs = get_runs(layout, acq, rec)
    
    events_list, bold_list, confounds_list, masks, trs = [], [], [], [], []

    for run in runs:
        ev_f = get_single(layout, acq, rec, run, suffix="events", extension=".tsv")
        events_list.append(pd.read_csv(ev_f.path, sep="\t")[["onset", "duration", "trial_type"]])

        bold_f = get_single(
            layout, acq, rec, run, space=SPACE, desc="preproc", suffix="bold", extension=".nii.gz",
        )
        bold_list.append(bold_f.path)
        trs.append(bold_f.get_metadata()["RepetitionTime"])

        mask_f = get_single(
            layout, acq, rec, run, space=SPACE, desc="brain", suffix="mask", extension=".nii.gz",
        )
        masks.append(mask_f.path)

        confounds_f = get_single(layout, acq, rec, run, desc="confounds", suffix="timeseries", extension=".tsv")
        confounds_list.append(read_confounds(confounds_f.path))

    assert len(set(trs)) == 1, f"inconsistent TR across runs for {acq}/{rec}: {trs}"
    mask_img = masks[0] if len(masks) == 1 else intersect_masks(
        masks, threshold=1)

    model = FirstLevelModel(
        t_r=trs[0],
        slice_time_ref=0.5,
        hrf_model="glover",
        drift_model=None,  # fMRIPrep's cosine confound regressors already high-pass filter
        high_pass=1/128,
        smoothing_fwhm=SMOOTHING_FWHM_MM,
        minimize_memory=True,
        standardize=True,
    )
        
    # Compute the number of dummy scans to skip for each sequence type. It
    # depends on the TR of each sequence.
    n_dummies = np.ceil(DUMMY_SECONDS / trs[0]).astype(int)

    # Load the BOLD images, skipping the dummy scans
    bold_niftis = [index_img(bl, slice(n_dummies, None)) for bl in bold_list]

    model = model.fit(
        run_imgs=bold_niftis,
        events=events_list,
        confounds=[cl[n_dummies:] for cl in confounds_list],
    )
    return model, runs


def run_combo(acq, rec):
    """Fit one acquisition/reconstruction's GLM and compute all its contrasts.

    This is the unit of work dispatched to a separate worker process per (acq, rec)
    combination. It builds its own local BIDSLayout rather than closing over the
    notebook's global one: joblib ships this function to workers by pickling it
    together with every global it references (there's no notebook module for a
    worker process to just re-import), and BIDSLayout can't be pickled -- it holds
    a live SQLite connection for its index. Rebuilding it here keeps this function
    self-contained and avoids that entirely; it's a small, one-off cost per worker
    next to a 30-70s GLM fit.

    Also unlike the old in-place loop, this can't mutate a shared t_maps /
    summary_rows dict -- worker processes don't share memory with the notebook
    kernel -- so it returns everything the caller needs instead, to be merged
    back in afterwards.
    """
    with threadpool_limits(1):  # each worker gets its own process; don't also let BLAS
                                 # multi-thread within it, or the 6-way process
                                 # parallelism oversubscribes the machine's cores
        layout = BIDSLayout(bids_root, validate=False, derivatives=True)
        model, runs = fit_glm(layout, acq, rec)

        con_thresh = st.norm.ppf(1 - ALPHA)

        contrast_results = {}
        rows = []
        for contrast_key, contrast_def in CONTRASTS.items():
            con_map = model.compute_contrast(
                [contrast_def] * len(runs), output_type="all", stat_type="t")
            
            n_suprathreshold = int((con_map['z_score'].get_fdata() > 
                                    con_thresh).sum())

            contrast_results[contrast_key] = (con_map['z_score'], con_thresh)
            rows.append({
                "acquisition": acq,
                "reconstruction": "nordic" if rec else "raw",
                "contrast": contrast_key,
                "n_runs": len(runs),
                "z_threshold": round(con_thresh, 3),
                "n_suprathreshold_voxels": n_suprathreshold,
            })

    return acq, rec, contrast_results, rows

## Fit all 6 GLMs and compute all four contrasts, in parallel

Each `(acquisition, reconstruction)` combination is an independent GLM fit, so
`run_combo` (fit + all 4 contrasts) runs as one task per combination via
`joblib.Parallel(backend="loky")` instead of a plain sequential loop. `loky` is
what nilearn/scikit-learn already use internally for this kind of thing, and
is more robust than `multiprocessing.Pool` when launched from inside a live
Jupyter kernel. Each worker pins its own BLAS thread count to 1
(`threadpoolctl.threadpool_limits`), since numpy's BLAS backend would
otherwise try to multi-thread every worker's linear algebra on top of the
6-way process parallelism already happening, oversubscribing the machine's
cores instead of speeding anything up.

`nilearn` will still warn that "the same contrast will be used for all N
runs" for the `2p0mm64ch` combinations (2 runs) &mdash; expected, since both
runs of a combination share the same design.

In [ ]:
combos = [(acq, rec) for acq in ACQUISITIONS for rec in RECONSTRUCTIONS]

con_maps = {}
summary_rows = []

parallel_results = Parallel(n_jobs=len(combos), return_as="generator_unordered")(
    delayed(run_combo)(acq, rec) for acq, rec in combos
)

for acq, rec, contrast_results, rows in parallel_results:
    label = f"{acq} / {'NORDIC' if rec else 'no NORDIC'}"
    print(f"Fitted {label}")
    for contrast_key, value in contrast_results.items():
        con_maps[(acq, rec, contrast_key)] = value
    summary_rows.extend(rows)

summary = pd.DataFrame(summary_rows)
summary

## Interactive HTML report

Each thresholded map gets its own standalone interactive viewer
(`nilearn.plotting.view_img`, saved via `save_as_html`). `index.html` wraps
all 24 (6 GLMs &times; 4 contrasts) in a single page with three independent
dropdowns &mdash; acquisition, reconstruction, contrast &mdash; that combine
to pick the matching viewer. The viewer is scaled (not scrolled) to fit
whatever space is available in the browser window, so nothing is ever
occluded or requires scrolling.

If your browser refuses to load the iframe from a `file://` URL, serve the
folder instead, e.g. `python -m http.server` from `glm_protocol_comparison/`
and open `http://localhost:8000/index.html`.

In [ ]:
t1w_path = get_t1w().path

html_names = []

for (acq, rec, contrast_key), (con_map, con_thresh) in con_maps.items():
    contrast_def = CONTRASTS[contrast_key]
    label = f"{acq} / {'NORDIC' if rec else 'no NORDIC'}"

    view = view_img(
        con_map, bg_img=t1w_path, threshold=con_thresh,
        title=f"{label}: {contrast_def} (t > {con_thresh:.2f}, p < {ALPHA} unc.)",
        width_view=1200,
    )
    html_name = f"{acq}_{'nordic' if rec else 'raw'}_{contrast_key}.html"
    view.save_as_html(HTML_DIR / html_name)
    html_names.append(html_name)

summary["html"] = html_names
summary

In [ ]:
def build_index_html(out_path: Path):
    """Write a standalone index.html whose acquisition/reconstruction/contrast dropdowns swap the embedded z-map viewer iframe."""
    acq_options = "\n".join(f'<option value="{a}">{a}</option>' for a in ACQUISITIONS)
    rec_options = "\n".join(
        f'<option value="{r}">{"NORDIC" if r == "nordic" else "raw"}</option>' for r in ["raw", "nordic"]
    )
    contrast_options = "\n".join(f'<option value="{c}">{c}</option>' for c in CONTRASTS)

    default_src = f"htmls/{ACQUISITIONS[0]}_raw_{next(iter(CONTRASTS))}.html"

    # nilearn's brainsprite viewer sizes its canvas from its container's clientWidth alone and
    # derives the height from a *fixed* ratio set by the MNI grid's slice counts (~2.45:1 here,
    # constant across all 24 files since they share the same res-2 MNI152 grid) -- it has no
    # notion of the available height, so the iframe itself must never be taller than the space
    # available, in either dimension, or its content overflows and grows a scrollbar. The iframe
    # box is given its native 1600x654 aspect ratio via `aspect-ratio`, capped on both axes by
    # `max-width`/`max-height: 100%` of its flex-centered container -- the browser shrinks it to
    # fit whichever axis is tighter while preserving the ratio, and recenters it automatically on
    # every resize, with no JS needed for sizing.
    html = f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>ARC-fMRI protocol comparison: inference vs. application</title>
<style>
  body {{ font-family: sans-serif; margin: 0; display: flex; flex-direction: column; height: 100vh;
          overflow: hidden; }}
  header {{ padding: 0.75rem 1rem; border-bottom: 1px solid #ccc; display: flex; gap: 1.5rem;
            align-items: center; flex-wrap: wrap; }}
  label {{ display: flex; gap: 0.4rem; align-items: center; }}
  select {{ font-size: 1rem; padding: 0.25rem; }}
  #viewerWrap {{ flex: 1; display: flex; align-items: center; justify-content: center; overflow: hidden; }}
  #viewer {{ width: 1600px; height: 654px; max-width: 100%; max-height: 100%;
             aspect-ratio: 1600 / 654; border: none; }}
</style>
</head>
<body>
<header>
  <label>Acquisition <select id="acq">{acq_options}</select></label>
  <label>Reconstruction <select id="rec">{rec_options}</select></label>
  <label>Contrast <select id="contrast">{contrast_options}</select></label>
</header>
<div id="viewerWrap">
  <iframe id="viewer" src="{default_src}"></iframe>
</div>
<script>
  function updateViewer() {{
    var acq = document.getElementById('acq').value;
    var rec = document.getElementById('rec').value;
    var contrast = document.getElementById('contrast').value;
    document.getElementById('viewer').src = 'htmls/' + acq + '_' + rec + '_' + contrast + '.html';
  }}
  document.getElementById('acq').addEventListener('change', updateViewer);
  document.getElementById('rec').addEventListener('change', updateViewer);
  document.getElementById('contrast').addEventListener('change', updateViewer);
</script>
</body>
</html>
"""
    with open(out_path, "w") as f:
        f.write(html)


index_path = OUT_DIR / "index.html"
build_index_html(index_path)
print(f"Interactive report: {index_path}")